# Tumour Classification — Breast Cancer Dataset

**Business problem:** Given measurements of a breast tumour cell nucleus, predict whether the tumour is **malignant (cancerous)** or **benign (non-cancerous)**.

**ML task:** Supervised binary classification  
**Dataset:** `sklearn.datasets.load_breast_cancer` — 569 samples, 30 numerical features, 2 classes  
**Metric:** ROC-AUC (accounts for class imbalance)

**Sections**
1. Data Exploration
2. Data Preprocessing
3. Feature Engineering
4. Model Training
5. Model Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
print('Ready.')

---
## 1. Data Exploration

In [ ]:
# Load dataset
raw = load_breast_cancer()
df  = pd.DataFrame(raw.data, columns=raw.feature_names)
df['target'] = raw.target          # 1 = benign, 0 = malignant

print('Shape:', df.shape)
print('Classes:', dict(zip(raw.target_names, df['target'].value_counts().sort_index())))
df.head(3)

In [ ]:
# Missing values and basic statistics
print('Missing values:', df.isna().sum().sum())
df.describe().round(2)

In [ ]:
# Class distribution
counts = df['target'].value_counts().sort_index()
plt.bar(['Malignant (0)', 'Benign (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
plt.title('Class Distribution')
plt.ylabel('Count')
for i, v in enumerate(counts.values):
    plt.text(i, v + 3, str(v), ha='center')
plt.tight_layout()
plt.show()
print(f'Malignant: {counts[0]}  |  Benign: {counts[1]}  |  Imbalance ratio: {counts[1]/counts[0]:.2f}')

In [ ]:
# Distribution of 4 key features by class
key_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area']
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, feat in zip(axes, key_features):
    for label, colour in [(0, '#e74c3c'), (1, '#2ecc71')]:
        ax.hist(df.loc[df['target'] == label, feat], bins=20,
                alpha=0.6, color=colour, label=raw.target_names[label])
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Class', fontsize=11)
plt.tight_layout()
plt.show()

---
## 2. Data Preprocessing

The dataset is split into train and test sets **before** any fitting to prevent data leakage.  
Scaling is done inside an `sklearn.Pipeline` so it is applied correctly inside every cross-validation fold.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

# 80 / 20 stratified split — done first, before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')
print(f'Train churn rate: {y_train.mean():.2%}  |  Test: {y_test.mean():.2%}')

---
## 3. Feature Engineering

Two domain-motivated features are added:
- **`worst_mean_ratio`** — ratio of the "worst" measurement to the "mean" measurement for radius. A high ratio means the largest observed cells are much bigger than average → stronger indicator of malignancy.
- **`compactness_index`** — product of mean perimeter and mean concavity, capturing irregular cell shape.

In [ ]:
def add_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X['worst_mean_ratio']   = X['worst radius'] / (X['mean radius'] + 1e-6)
    X['compactness_index']  = X['mean perimeter'] * X['mean concavity']
    return X

X_train_eng = add_features(X_train)
X_test_eng  = add_features(X_test)

print('Features after engineering:', X_train_eng.shape[1])
X_train_eng[['worst radius', 'mean radius', 'worst_mean_ratio',
              'mean perimeter', 'mean concavity', 'compactness_index']].head(3)

---
## 4. Model Training

Three models are trained:

| Model | Notes |
|-------|-------|
| Logistic Regression | Linear baseline |
| Random Forest | Ensemble, handles non-linearity |
| SVM | Effective in high-dimensional spaces |

Each is wrapped in a `Pipeline` (StandardScaler → classifier) so scaling never leaks across CV folds.  
Hyperparameters are tuned with `GridSearchCV` using 5-fold stratified cross-validation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Baseline cross-validation (default hyperparameters) ──────────────────────
# Run before tuning to confirm all three models are worth tuning.
base_models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()), ('clf', RandomForestClassifier(random_state=RANDOM_STATE))]),
    'SVM':                 Pipeline([('sc', StandardScaler()), ('clf', SVC(probability=True, random_state=RANDOM_STATE))])
}

print('Baseline 5-fold CV (ROC-AUC):')
for name, pipe in base_models.items():
    scores = cross_val_score(pipe, X_train_eng, y_train, cv=cv, scoring='roc_auc')
    print(f'  {name:<22} mean={scores.mean():.4f}  std={scores.std():.4f}')

In [ ]:
# ── Hyperparameter tuning — Logistic Regression ───────────────────────────────
lr_grid = {'clf__C': [0.01, 0.1, 1, 10], 'clf__penalty': ['l1', 'l2'],
           'clf__solver': ['liblinear']}

lr_search = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    lr_grid, cv=cv, scoring='roc_auc', refit=True, n_jobs=-1
)
lr_search.fit(X_train_eng, y_train)
print('LR best params:', lr_search.best_params_)
print('LR best CV AUC:', round(lr_search.best_score_, 4))

In [ ]:
# ── Hyperparameter tuning — Random Forest ────────────────────────────────────
rf_grid = {'clf__n_estimators': [50, 100], 'clf__max_depth': [None, 5, 10],
           'clf__min_samples_split': [2, 5]}

rf_search = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('clf', RandomForestClassifier(random_state=RANDOM_STATE))]),
    rf_grid, cv=cv, scoring='roc_auc', refit=True, n_jobs=-1
)
rf_search.fit(X_train_eng, y_train)
print('RF best params:', rf_search.best_params_)
print('RF best CV AUC:', round(rf_search.best_score_, 4))

In [ ]:
# ── Hyperparameter tuning — SVM ───────────────────────────────────────────────
svm_grid = {'clf__C': [0.1, 1, 10], 'clf__kernel': ['rbf', 'linear'],
            'clf__gamma': ['scale', 'auto']}

svm_search = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('clf', SVC(probability=True, random_state=RANDOM_STATE))]),
    svm_grid, cv=cv, scoring='roc_auc', refit=True, n_jobs=-1
)
svm_search.fit(X_train_eng, y_train)
print('SVM best params:', svm_search.best_params_)
print('SVM best CV AUC:', round(svm_search.best_score_, 4))

In [ ]:
# ── Select best model based on CV score ───────────────────────────────────────
# Model selection uses only CV scores from the training set — test set not touched yet.
results = {
    'Logistic Regression': lr_search,
    'Random Forest':       rf_search,
    'SVM':                 svm_search
}

summary = pd.DataFrame({
    'CV ROC-AUC': {n: s.best_score_ for n, s in results.items()}
}).round(4)
print(summary.to_string())

best_name  = summary['CV ROC-AUC'].idxmax()
best_model = results[best_name].best_estimator_
print(f'\nSelected model: {best_name}')

---
## 5. Model Evaluation

The test set is evaluated **once**, at the very end, after model selection is complete.

In [ ]:
y_pred = best_model.predict(X_test_eng)
y_prob = best_model.predict_proba(X_test_eng)[:, 1]

print(f'=== Test Set Results — {best_name} ===')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=raw.target_names))

In [ ]:
# Confusion matrix and ROC curves for all three models
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=raw.target_names,
    cmap='Blues', ax=axes[0], colorbar=False
)
axes[0].set_title(f'Confusion Matrix — {best_name}')

for name, search in results.items():
    model = search.best_estimator_
    prob  = model.predict_proba(X_test_eng)[:, 1]
    RocCurveDisplay.from_predictions(
        y_test, prob,
        name=f'{name} (AUC={roc_auc_score(y_test, prob):.3f})',
        ax=axes[1]
    )
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('ROC Curves — Test Set')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()